In [1]:
import pandas as pd
import numpy as np

print("--- Phase 2: Exploratory Data Analysis (Training Fold Only) ---")

# 1. LOAD ONLY THE 80% TRAINING DATA
# (This physically guarantees zero data leakage)
file_path = "logs_80percent.csv" # Update to your actual filename

# We can safely load this without chunking now because we downcasted in Step 1
df_train = pd.read_csv(file_path)
print(f"Training Data Loaded Successfully. Shape: {df_train.shape[0]} rows, {df_train.shape[1]} columns.")

# Separate features and target (Assuming 'LabelEnc' is the target)
target_col = 'LabelEnc'
X_train = df_train.drop(columns=[target_col])
y_train = df_train[target_col]

# 2. EXACT CLASS IMBALANCE CALCULATION
print("\n--- 1. Class Imbalance Verification ---")
class_counts = y_train.value_counts()
class_percentages = y_train.value_counts(normalize=True) * 100

imbalance_df = pd.DataFrame({
    'Count': class_counts,
    'Percentage (%)': class_percentages
})
print(imbalance_df)

# 3. ZERO-VARIANCE THRESHOLDING
print("\n--- 2. Zero-Variance Feature Identification ---")
# Find columns where there is only 1 unique value across the entire dataset
unique_counts = X_train.nunique()
zero_variance_cols = unique_counts[unique_counts <= 1].index.tolist()

print(f"Found {len(zero_variance_cols)} columns with zero variance (no information gain).")
if len(zero_variance_cols) > 0:
    print("Sample of zero-variance columns to drop:", zero_variance_cols[:5])
    # Drop them immediately to save RAM and computation time later
    X_train = X_train.drop(columns=zero_variance_cols)
    print(f"Zero-variance columns dropped. New Feature Shape: {X_train.shape}")

# 4. SPARSITY / MISSING VALUE AUDIT
print("\n--- 3. Missing Value Audit ---")
missing_data = X_train.isnull().sum()
missing_data = missing_data[missing_data > 0].sort_values(ascending=False)

if missing_data.empty:
    print("Dataset is perfectly dense. No missing values detected.")
else:
    print(f"Found {len(missing_data)} columns with missing values:")
    print(missing_data.head())

--- Phase 2: Exploratory Data Analysis (Training Fold Only) ---
Training Data Loaded Successfully. Shape: 2264594 rows, 177 columns.

--- 1. Class Imbalance Verification ---
            Count  Percentage (%)
LabelEnc                         
0         1818477       80.300354
4          184858        8.162964
10         127144        5.614428
2          102421        4.522709
3            8234        0.363597
7            6350        0.280403
11           4718        0.208338
6            4637        0.204761
5            4399        0.194251
1            1573        0.069461
12           1206        0.053255
14            522        0.023050
9              29        0.001281
13             17        0.000751
8               9        0.000397

--- 2. Zero-Variance Feature Identification ---
Found 0 columns with zero variance (no information gain).

--- 3. Missing Value Audit ---
Dataset is perfectly dense. No missing values detected.
